<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### ML task type: Ranking

I would frame content refresh prioritization as a ranking task. The goal is to rank pages from highest to lowest priority for content refresh review, so that an SEO or content team can review the most important pages first. Ranking fits the decision because the team has limited time and needs to know which pages should come first rather than only assigning a yes/no label.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the ML task type for this lane

# Setup: clone your GitHub repository and load the dataset

import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/vaishnavikabbe/AIML.git"
REPO_DIR = "/content/AIML"

# Clone the repository if it is not already available
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )
else:
    print("Repository already exists.")

# Move into the repository
os.chdir(REPO_DIR)

# Load the dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Current folder:", os.getcwd())
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Cloning repository...

Dataset loaded successfully.
Current folder: /content/AIML
Rows: 30000
Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / proxy

The main observed outcome I will use is `trend_direction`. In particular, pages with a `down` trend represent pages whose search performance is declining in the observed data. I will use this outcome as a proxy for refresh priority rather than claiming that it directly represents whether a page must be refreshed. The model will use available search-performance and content signals to rank pages that may deserve review.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter dataset and inspect the observed outcome

# Load the starter dataset

# Check the observed outcome

print("Dataset shape:", df.shape)

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts())

Dataset shape: (30000, 44)

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric

I will use Precision@K as the main success metric because the goal is to prioritize a limited number of pages for content refresh review. Precision@K measures how many of the top K pages selected by the model are actually relevant for review.

A higher Precision@K means the team spends its limited review time on more useful pages. I will use Precision@50 as the main metric because reviewing the top 50 pages gives a practical and easy-to-interpret shortlist.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the success metric for the ML task

k = 50

print("Success metric: Precision@50")
print("K value:", k)
print("Goal: A higher Precision@50 means more of the top 50 pages are relevant for review.")

Success metric: Precision@50
K value: 50
Goal: A higher Precision@50 means more of the top 50 pages are relevant for review.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis

The unit of analysis is one page. Each row in the dataframe represents one anonymized webpage and contains its search-performance and content-related signals.

I will use these page-level observations to understand which pages may be suitable for content refresh review and to build a model that can prioritize them.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the dataset and show the unit of analysis

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Unit of analysis: one page")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nSample of the dataframe:")
display(df.head())


Unit of analysis: one page
Rows: 30000
Columns: 44

Sample of the dataframe:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML beats a fixed rule

A fixed rule such as "refresh every page with low impressions" would be too simple because page performance can depend on several signals at the same time. Search performance, content characteristics, and other page-level signals may interact in ways that are difficult to capture with one if-statement.

ML can combine multiple signals and learn patterns from the observed data. This makes it useful for creating a ranked shortlist of pages for review instead of relying on one manually chosen threshold.

The ML result will still be used as decision-support, not as an automatic decision.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show several signals that can be used together

candidate_features = [
    "search_volume",
    "impressions_90d",
    "word_count"
]

available_features = [col for col in candidate_features if col in df.columns]

print("Candidate signals available:")
print(available_features)

print("\nNumber of signals:", len(available_features))


Candidate signals available:
['search_volume', 'impressions_90d', 'word_count']

Number of signals: 3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.